# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = "../../data/processed/refresh_feature_vector.csv"

df = pd.read_csv(data_path)

# Signals: visibility, SERP slip (avg_position > 20, ignoring 0 which means no data), and staleness
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["slip"] = ((df["avg_position"] > 20) & (df["avg_position"] > 0)).astype(int)
df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["thin"] = ((df["word_count"] > 0) & (df["word_count"] < 1200)).astype(int)

# Score weighting position slip and staleness by impression volume
df["score"] = df["visible"] * (df["slip"] * df["impressions_90d"] + df["stale"] * df["impressions_90d"] * 0.3)

# Transparent reason codes
def get_reason(r):
    if r["slip"] and r["stale"]: return "STALE_AND_SLIPPING"
    if r["slip"]: return "POSITION_SLIP_ONLY"
    if r["stale"] and r["visible"]: return "STALE_HIGH_VISIBILITY"
    if r["thin"] and r["visible"]: return "THIN_CONTENT_OPPORTUNITY"
    return "LOW_PRIORITY"

def get_action(r):
    if r["reason_code"] == "THIN_CONTENT_OPPORTUNITY": return "EXPAND_AND_REFRESH"
    if r["score"] > 0: return "REFRESH"
    return "MONITOR"

df["reason_code"] = df.apply(get_reason, axis=1)
df["action"] = df.apply(get_action, axis=1)

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue[["content_id", "score", "reason_code", "action", "impressions_90d", "avg_position", "days_since_last_update"]].head(10)

,content_id,score,reason_code,action,impressions_90d,avg_position,days_since_last_update
0,content_2dba2b1f9536,576464.2,STALE_AND_SLIPPING,REFRESH,443434,27.9,104
1,content_2cb567c3c89b,497727.0,POSITION_SLIP_ONLY,REFRESH,497727,22.2,48
2,content_b28d1efd668f,372590.4,STALE_AND_SLIPPING,REFRESH,286608,26.2,104
3,content_813e88069237,303629.3,STALE_AND_SLIPPING,REFRESH,233561,26.2,104
4,content_b511d4bc4ad2,267689.5,STALE_AND_SLIPPING,REFRESH,205915,27.9,104
5,content_f02b48f88241,235968.2,STALE_AND_SLIPPING,REFRESH,181514,25.8,104
6,content_05e9b4cd9ccf,232702.6,STALE_AND_SLIPPING,REFRESH,179002,22.1,104
7,content_ff94c9b6b411,228566.0,POSITION_SLIP_ONLY,REFRESH,228566,27.4,20
8,content_66b4046cc144,217415.0,POSITION_SLIP_ONLY,REFRESH,217415,26.6,20
9,content_a023517539fe,214047.0,POSITION_SLIP_ONLY,REFRESH,214047,85.8,20


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use
- **Who uses it:** Content strategists and SEO editors managing large portfolios.
- **For what:** Prioritizing which decaying, high-demand pages to audit and refresh first.

### Known Limits & Boundary Conditions
- **Decision-support, not causal:** The score highlights observed historical risk; it does not guarantee that refreshing will restore rankings.
- **Rate scales:** Rates (`ctr`, `engagement_rate`, `scroll_rate`) are $\times 100$ percentages (`ctr=0.76` means $0.76\%$).
- **Missing positions:** `avg_position = 0` indicates unranked/missing SERP data, not rank zero.
- **Low search volume:** Pages with $<500$ impressions have too little search signal for reliable decay classification.

In [9]:
print(f"Total pages: {len(df):,}")
print(f"Pages with avg_position == 0 (no SERP data): {(df['avg_position'] == 0).sum():,} ({(df['avg_position'] == 0).mean()*100:.1f}%)")
print(f"High-visibility candidates (impressions >= 500): {(df['impressions_90d'] >= 500).sum():,} ({(df['impressions_90d'] >= 500).mean()*100:.1f}%)")
print(f"Top 50 queue decline rate: {queue.head(50)['is_declining_label'].mean():.3f}")

Total pages: 30,000
Pages with avg_position == 0 (no SERP data): 1,205 (4.0%)
High-visibility candidates (impressions >= 500): 16,726 (55.8%)
Top 50 queue decline rate: 0.560


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Checklist
1. **Search Intent:** Has user search intent changed or did new SERP features (AI Overviews, Snippets) alter CTR?
2. **Seasonality:** Is the traffic drop cyclical rather than true content decay?
3. **Business Value:** Does this page drive business conversions or is it low-value informational traffic?

### The No-Go List (Never Automate)
- **Legal & Compliance:** Privacy, Terms, and Policy pages must never be auto-refreshed.
- **Brand & Core Navigation:** Fluctuations on brand queries are not content decay problems.
- **Recently Updated (<30d):** Pages refreshed within 30 days must be held for cooldown to allow search crawlers time to index.

In [10]:
recent_updates_top100 = (queue.head(100)["days_since_last_update"] < 30).sum()
print(f"No-go check: {recent_updates_top100} of top 100 pages were updated <30 days ago (held for cooldown).")

No-go check: 17 of top 100 pages were updated <30 days ago (held for cooldown).


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Retrain & Monitoring Triggers
- **Data Drift:** Median portfolio CTR or impression distribution shifts $>20\%$ (e.g. from search layout shifts).
- **Model Performance Decay:** Holdout Precision@50 drops below $0.55$ on new client batches.
- **Calibration Disconnect:** Observed decline rate in the top queue falls below portfolio base rate (~$51\%$).
- **Cadence:** Quarterly scheduled retraining or immediate retraining after confirmed Google Core Updates.

In [11]:
print("Monitoring baselines:")
print(f"- Portfolio median impressions: {df['impressions_90d'].median():.0f}")
print(f"- Portfolio median CTR: {df['ctr'].median():.2f}%")
print(f"- Portfolio base decline rate: {df['is_declining_label'].mean():.3f}")
print(f"- Top 50 queue decline rate: {queue.head(50)['is_declining_label'].mean():.3f}")

Monitoring baselines:
- Portfolio median impressions: 731
- Portfolio median CTR: 0.07%
- Portfolio base decline rate: 0.542
- Top 50 queue decline rate: 0.560


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [12]:
out_dir = Path("../../work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

export_cols = [
    "content_id",
    "score",
    "reason_code",
    "action",
    "impressions_90d",
    "avg_position",
    "days_since_last_update",
    "is_declining_label",
]

queue[export_cols].to_csv(out_dir / "action_playbook_queue.csv", index=False)
print(f"Exported queue to {out_dir / 'action_playbook_queue.csv'} ({len(queue):,} rows)")
print(f"\nAction breakdown:\n{queue['action'].value_counts()}")

Exported queue to ../../work/outputs/action_playbook_queue.csv (30,000 rows)

Action breakdown:
action
MONITOR               20933
REFRESH                9053
EXPAND_AND_REFRESH       14
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.